# LCU Select and Prepare Oracles

Adapted from Qualtran v0.7.0, commit bcb17bb25360ed68ce3bc084645db3e929799389.
Copyright Google LLC. SPDX-License-Identifier: Apache-2.0.

In [ ]:
from dataclasses import asdict
from IPython.display import Markdown, SVG, display

from examples.algorithms.field_addition_catalog import field_signature_svg
from examples.algorithms.hubbard_lcu_catalog import HubbardLCUExample, hubbard_lcu_circuit
from flagquantum.compiler._component_trace.call_graph import modeled_call_graph
from flagquantum.compiler._component_trace.resource_model import estimate_component_resources
from flagquantum.core.ir.program.signatures import Signature
from flagquantum.drawer.call_graph import (
    export_modeled_call_graph_dot,
    format_call_graph_markdown,
    render_call_graph_svg,
)
from flagquantum.qec.resources.estimation import count_emitted_instructions

def show_signatures(*examples):
    display(SVG(field_signature_svg([
        (name, Signature.build(**dict(layout)), {}) for name, layout in examples
    ])))

def show_calls(component):
    graph = modeled_call_graph(component, max_depth=1)
    display(SVG(render_call_graph_svg(export_modeled_call_graph_dot(component, max_depth=1))))
    display(Markdown(format_call_graph_markdown(
        graph.root, graph.direct_calls, graph.leaf_counts, title=lambda node: node.name,
    )))

## `SelectBlockEncoding`
LCU based block encoding using SELECT and PREPARE oracles.

Builds the block encoding via
$$
    B[H] = \mathrm{SELECT}
$$

$$
    \mathrm{SELECT} |l\rangle_a|\psi\rangle_s = |l\rangle_a U_l |\psi\rangle_s.
$$

The Hamiltonian can be extracted via

$$
    \langle G | B[H] | G \rangle = H / \alpha
$$

where

$$
    |G\rangle = \mathrm{PREPARE} |0\rangle_a = \sum_l \sqrt{\frac{w_l}{\alpha}} |l\rangle_a,
$$

The ancilla register is at least of size $\log L$.

In our implementations we typically split the ancilla registers into
selection registers (i.e.  the $l$ registers above) and junk registers which
are extra qubits needed by state preparation but not controlled upon during
SELECT.

#### Parameters
 - `select`: The operation implementing the `SelectOracle` interface.
 - `prepare`: The operation implementing the `PrepareOracle` interface. 

#### Registers
 - `selection`: The combined selection register.
 - `junk`: Additional junk registers not prepared upon.
 - `system`: The combined system register. 

#### References
 - [Hamiltonian Simulation by Qubitization](https://quantum-journal.org/papers/q-2019-07-12-163/).     Low et al. 2019. Sec 3.1, page 7 and 8 for high level overview and definitions. A     block encoding is called a standard form encoding there.
 - [The power of block-encoded matrix powers: improved regression techniques via faster Hamiltonian simulation](https://arxiv.org/abs/1804.01973).     Chakraborty et al. 2018. Definition 3 page 8.


In [ ]:
from flagquantum.algorithms.primitives.select_block_encoding import SelectBlockEncoding

### Example Instances

In [ ]:
named = HubbardLCUExample()
select_block = named.select_block
assert isinstance(select_block, SelectBlockEncoding)
assert select_block.alpha == 72

In [ ]:
packed = HubbardLCUExample(packed=True)
black_box_select_block = packed.select_block
assert black_box_select_block.alpha == select_block.alpha

#### Graphical Signature

In [ ]:
show_signatures(
    ("Named SELECT block", (*select_block.selection_layout, *select_block.target_layout)),
    ("Packed SELECT block",
     (*black_box_select_block.selection_layout, *black_box_select_block.target_layout)),
)

### Call Graph

In [ ]:
show_calls(select_block.resource_model(named.select_model()))

## `LCUBlockEncoding`
LCU based block encoding using SELECT and PREPARE oracles.

Builds the standard block encoding from an LCU as
$$
    B[H] = \mathrm{PREPARE}^\dagger \cdot \mathrm{SELECT} \cdot \mathrm{PREPARE},
$$
where
$$
    \mathrm{PREPARE} |0\rangle_a = \sum_l \sqrt{\frac{w_l}{\alpha}} |l\rangle_a,
$$
and
$$
    \mathrm{SELECT} |l\rangle_a|\psi\rangle_s = |l\rangle_a U_l |\psi\rangle_s.
$$

The Hamiltonian can be extracted via
$$
    \langle 0 | B[H] | 0 \rangle = H / \alpha,
$$
This differs from the `SelectBlockEncoding` which uses Prepare for the signal state, while here
it is the identity operator.

The ancilla register is at least of size $\log L$.

In our implementations we typically split the ancilla registers into
selection registers (i.e.  the $l$ registers above) and junk registers which
are extra qubits needed by state preparation but not controlled upon during
SELECT.

#### Parameters
 - `select`: The operation implementing the `SelectOracle` interface.
 - `prepare`: The operation implementing the `PrepareOracle` interface. 

#### Registers
 - `selection`: The combined selection register.
 - `junk`: Additional junk registers not prepared upon.
 - `system`: The combined system register. 

#### References
 - [Hamiltonian Simulation by Qubitization](https://quantum-journal.org/papers/q-2019-07-12-163/).     Low et al. 2019. Sec 3.1, page 7 and 8 for high level overview and definitions. A     block encoding is called a standard form encoding there.
 - [The power of block-encoded matrix powers: improved regression techniques via faster Hamiltonian simulation](https://arxiv.org/abs/1804.01973).     Chakraborty et al. 2018. Definition 3 page 8.


In [ ]:
from flagquantum.algorithms.primitives.lcu_block_encoding import LCUBlockEncoding

### Example Instances

In [ ]:
lcu_block = named.lcu_block()
assert isinstance(lcu_block, LCUBlockEncoding)
assert (lcu_block.ancilla_bitsize, lcu_block.resource_bitsize, lcu_block.system_bitsize) == (12, 2, 18)

In [ ]:
black_box_lcu_block = packed.lcu_block()
assert black_box_lcu_block.signal_state.selection_layout == (("selection", 12),)

#### Graphical Signature

In [ ]:
def lcu_layout(block):
    return (*block.selection_layout, *block.junk_layout, *block.target_layout)

show_signatures(
    ("Named LCU block", lcu_layout(lcu_block)),
    ("Packed LCU block", lcu_layout(black_box_lcu_block)),
)

### Call Graph

In [ ]:
show_calls(named.lcu_model())

## `SelectOracle`
A SELECT oracle receives bound selection and system registers.

The action of a SELECT oracle on a selection register $|l\rangle$ and target register
$|\Psi\rangle$ can be defined as:

$$
    \mathrm{SELECT} = \sum_{l}|l \rangle \langle l| \otimes U_l
$$

In other words, the `SELECT` oracle applies $l$'th unitary $U_l$ on the target register
$|\Psi\rangle$ when the selection register stores integer $l$.

$$
    \mathrm{SELECT}|l\rangle |\Psi\rangle = |l\rangle U_{l}|\Psi\rangle
$$

In [ ]:
from flagquantum.algorithms.primitives.hubbard_select import SelectHubbard
print(SelectHubbard(3).layout)

## `PrepareOracle`
A PREPARE oracle receives bound selection and junk registers.

Given a set of coefficients $\{c_0, c_1, ..., c_{N - 1}\}$, the PREPARE oracle is used to encode
the coefficients as amplitudes of a state $|\Psi\rangle = \sum_{l=0}^{N-1} \sqrt{\frac{c_l}{\lambda}} |l\rangle$
where $\lambda = \sum_l |c_l|$, using a selection register $|l\rangle$. In order to prepare such
a state, the PREPARE circuit is also allowed to use a junk register that is entangled with
selection register.

Thus, the action of a PREPARE circuit on an input state $|0\rangle$ can be defined as:

$$
    \mathrm{PREPARE} |0\rangle = \sum_{l=0}^{N-1} \sqrt{ \frac{c_l}{\lambda} } |l\rangle |\mathrm{junk}_l\rangle
$$

In [ ]:
from flagquantum.algorithms.primitives.hubbard_prepare import PrepareHubbard
print(PrepareHubbard(3, 1, 4).layout)

## `BlackBoxSelect`
A 'black box' Select operation.

The `SELECT` operation applies the $l$'th unitary $U_{l}$ on the system register
when the selection register stores integer $l$.
When implementing specific `SelectOracle` operations, it is helpful to have multiple selection
registers each with semantic meaning. For example: you could have spatial or spin coordinates
on different, named registers. The `SelectOracle` interface encourages this. `BlackBoxSelect`
uses the properties on the `SelectOracle` interface to provide a "black box" view of a select
operation that just has a selection and system register.
During decomposition, this operation uses ordered register binding to partition
and route the parts of the unified selection register to the `Select` operation.

#### Parameters
 - `select`: The operation implementing the `SelectOracle` interface. 

#### Registers
 - `selection`: The combined selection register
 - `system`: The combined system register


In [ ]:
packed_two = HubbardLCUExample(side=2, packed=True)

### Example Instances

In [ ]:
black_box_select = packed_two.select_block

#### Graphical Signature

In [ ]:
show_signatures(("Packed 2 × 2 SELECT",
                  (*black_box_select.selection_layout, *black_box_select.target_layout)))

### Call Graph

In [ ]:
show_calls(packed_two.select_model())

## `BlackBoxPrepare`
Provide a black-box interface to `Prepare` operations.

This wrapper uses the native register partition to combine descriptive selection
registers into one register named "selection".

#### Parameters
 - `prepare`: The operation following the `Prepare` interface to wrap. 

#### Registers
 - `selection`: selection register.
 - `junk`: Additional junk registers not prepared upon.


In [ ]:
from flagquantum.algorithms.primitives.black_box_prepare import BlackBoxPrepare

### Example Instances

In [ ]:
black_box_prepare = packed_two.prepare_oracle
assert isinstance(black_box_prepare, BlackBoxPrepare)

#### Graphical Signature

In [ ]:
show_signatures(("Packed 2 × 2 PREPARE",
                  (*black_box_prepare.selection_layout, *black_box_prepare.junk_layout)))

### Call Graph

In [ ]:
show_calls(packed_two.prepare_model())

### Append and inspect the Hubbard LCU

The named and packed forms bind the same physical wires. Compile both for the
positive-controlled 3 × 3 case and compare their complete instruction streams.
The classical slots and temporary work can be reused across sequential stages.

In [ ]:
named_circuit = hubbard_lcu_circuit(named, control=1)
packed_circuit = hubbard_lcu_circuit(packed, control=1)
assert named_circuit.to_ir().instructions == packed_circuit.to_ir().instructions
physical = count_emitted_instructions(named_circuit.to_ir())
logical = estimate_component_resources(named.lcu_model(control=1))
print("Emitted T/T-dagger instructions", physical.t_count)
print("Emitted measurements", physical.unconditional["measure"])
print("Source gate model", asdict(logical.gates))
print("Source peak logical qubits", logical.peak_logical_qubits)